# Nesli — Veri kümesi, ground truth ve RGB/HSV başlangıç analizi

Branch: `feat/dataset-groundtruth`

Bu notebook altı özgün görüntünün kaynak/etiket bütünlüğünü denetler, development görüntülerinde renk eşiklerini kalibre eder ve dondurulmuş ayarlarla dört control görüntüsünü ölçer. Renk maskesi taban çizgisidir; eğitilmiş baret dedektörü değildir.


## 1. Ortam ve ortak veri

Notebook, depo kökünü çalışma dizininden bulur. Kaynak CSV, ground truth JSON ve görüntü dosyalarını birleştirip piksel boyutlarını denetler.


In [1]:
from pathlib import Path
import csv
import json
import sys
import cv2

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook'u klonlanan depo içinden açın.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.common import load_dataset
from scripts.run_nesli_dataset_analysis import run_analysis, format_summary

records = load_dataset()
assert len(records) == 6, f"6 kaynak/etiket bekleniyordu, bulunan: {len(records)}"
assert {row["image_id"] for row in records} == {"G1", "G2", "K1", "K2", "K3", "K4"}

print("id split size condition class head_roi helmets")
for item in records:
    image = cv2.imread(str(ROOT / item["relative_path"]))
    assert image is not None, f"Açılamayan görüntü: {item['relative_path']}"
    height, width = image.shape[:2]
    assert (width, height) == (int(item["width"]), int(item["height"]))
    print(f"{item['image_id']:>2} {item['split']:<11} {width}x{height:<5} "
          f"{item['condition']:<38} {item['ground_truth_class']:<17} "
          f"{str(item['head_roi']):<22} {len(item['helmet_boxes'])}")

class_counts = {}
for item in records:
    class_counts[item["ground_truth_class"]] = class_counts.get(item["ground_truth_class"], 0) + 1
print("Class counts:", class_counts)


id split size condition class head_roi helmets
G1 development 256x256   daylight_site_yellow_equipment_distractor no_helmet         [99, 28, 140, 108]     0
G2 development 1023x683   bright_studio_like                     helmet_on_head    [280, 14, 466, 278]    1
K1 control     256x256   daylight_site_head_cropped             uncertain         None                   0
K2 control     1300x957   night_low_light_deep_shadows           helmet_on_head    [976, 90, 1090, 236]   1
K3 control     2121x1414  distant_small_heads                    helmet_on_head    [1092, 826, 1140, 887] 2
K4 control     1300x957   bright_studio_helmet_held              helmet_off_head   [430, 0, 704, 252]     1
Class counts: {'no_helmet': 1, 'helmet_on_head': 3, 'uncertain': 1, 'helmet_off_head': 1}


## 2. Deney öncesi tahmin ve dondurulan ayarlar

Tahminler control maskeleri çalıştırılmadan önce `docs/predictions.csv` içine kaydedildi. K1'de baş görünmediği için `uncertain`, K4'te baret elde olduğu için `helmet_off_head` bekleniyor. Karanlık K2 ve küçük/uzak K3 örneklerinde renk maskesinin kaçırma yapması olası.

RGB/HSV aralıkları ve ortak sınıf eşiği yalnızca G1/G2 development verisinden belirlenmiştir. Eşikler control ölçümü öncesinde dondurulur; sonuç görüldükten sonra değiştirilmez.

Belirsizlik karşılaştırması: K2’de baş ve baret görünür, dolayısıyla gerçek sınıf kanıta göre belirlenebilir; düşük ışık sadece renk yöntemini zorlar. K1’de başlar kadraj dışında kaldığı için baret takılı mı değil mi görülemez; bu nedenle head_roi null ve sınıf uncertain. Sonraki kontrol, tam başları içeren bir görüntüyle K1’deki kadraj belirsizliğini gidermektir.


In [2]:
configs = {
    method: json.loads((ROOT / f"configs/{method}_config.json").read_text(encoding="utf-8"))
    for method in ("rgb", "hsv")
}
decision = json.loads((ROOT / "configs/experiment.json").read_text(encoding="utf-8"))["decision_rule"]
assert all(c["frozen"] and c["status"] == "configured" for c in configs.values())
assert decision["frozen"] and decision["status"] == "configured"
for method, config in configs.items():
    print(f"{method.upper()}: {config['color_space']} lower={config['lower']} upper={config['upper']}")
print("Shared rule:", decision["insufficient_evidence_rule"])


RGB: RGB lower=[180, 140, 0] upper=[255, 255, 120]
HSV: HSV_OPENCV_8BIT lower=[18, 80, 80] upper=[50, 255, 255]
Shared rule: If head_roi is null, predict uncertain. With a visible head, mask area below 2.5 percent predicts no_helmet; otherwise head overlap of at least 5 percent predicts helmet_on_head, and lower overlap predicts helmet_off_head. These thresholds were selected on G1/G2 and frozen before control evaluation.


## 3. Ölçüm

Kod G1/G2 kalibrasyon ölçülerinin dondurulmuş eşiklerle ayrıştığını doğrular; sonra K1–K4 için özgün boyutta RGB ve HSV maskelerini üretir. `mask_area_percent` tüm görüntüye, `head_overlap_percent` baş ROI içindeki seçili piksellere göre hesaplanır. ROI yoksa `N/A` yazılır.

Maskeler `results/masks/` altına, görsel ground truth kutu önizlemeleri `annotations/previews/` altına kaydedilir. Tahmin sınıfı sabitlenmiş renk alanı/baş örtüşmesi kuralıyla çıkarılır.


In [3]:
analysis = run_analysis(write_outputs=True)
print(format_summary(analysis))


DEVELOPMENT CALIBRATION (G1/G2; thresholds fixed before controls)
image method area_percent head_overlap_percent ground_truth
G1 rgb 0.8652 0.0305 no_helmet
G1 hsv 2.3224 0.0000 no_helmet
G2 rgb 9.7335 24.5560 helmet_on_head
G2 hsv 3.7990 23.2201 helmet_on_head

CONTROL RESULTS (K1-K4)
image method condition area_percent head_overlap_percent prediction ground_truth correct
K1 rgb daylight_site_head_cropped 0.0000 N/A uncertain uncertain true
K1 hsv daylight_site_head_cropped 0.0000 N/A uncertain uncertain true
K2 rgb night_low_light_deep_shadows 0.0831 1.3278 no_helmet helmet_on_head false
K2 hsv night_low_light_deep_shadows 2.0203 16.0478 no_helmet helmet_on_head false
K3 rgb distant_small_heads 0.0266 12.0219 no_helmet helmet_on_head false
K3 hsv distant_small_heads 0.0394 14.8566 no_helmet helmet_on_head false
K4 rgb bright_studio_helmet_held 3.4250 3.6612 helmet_off_head helmet_off_head true
K4 hsv bright_studio_helmet_held 3.6880 0.0391 helmet_off_head helmet_off_head true

CLASS 

## 4. Hata analizi ve sonuç dosyaları

`results/results.csv` içinde iki yöntem × dört control görüntüsü için sekiz ölçülmüş satır vardır. Kutu düzeyinde bir dedektör değerlendirmesi yapılmadığından `tp/fp/fn` dürüstçe `N/A` bırakılır. Sınıf doğruluğu ayrı hesaplanır.


In [4]:
with (ROOT / "results/results.csv").open(encoding="utf-8", newline="") as handle:
    result_rows = [row for row in csv.DictReader(handle)
                   if row["experiment_id"] == "nesli_rgb_hsv_baseline"]
assert len(result_rows) == 8
print("method rows correct errors")
for method in ("rgb_color_mask", "hsv_color_mask"):
    rows = [r for r in result_rows if r["method"] == method]
    correct = sum(r["correct"] == "true" for r in rows)
    errors = [r["image_id"] for r in rows if r["correct"] != "true"]
    print(f"{method:<18} {len(rows)} {correct}/4 {errors}")

for row in result_rows:
    if row["correct"] != "true":
        reason = {
            "K2": "low-light night scene; the frozen color mask area is below the class threshold",
            "K3": "small/distant helmets; total color-mask area is below the class threshold",
        }.get(row["image_id"], "inspect the saved mask and ROI")
        print(f"{row['method']} {row['image_id']}: predicted {row['predicted_class']}, "
              f"ground truth {row['ground_truth_class']}; {reason}.")


method rows correct errors
rgb_color_mask     4 2/4 ['K2', 'K3']
hsv_color_mask     4 2/4 ['K2', 'K3']
rgb_color_mask K2: predicted no_helmet, ground truth helmet_on_head; low-light night scene; the frozen color mask area is below the class threshold.
rgb_color_mask K3: predicted no_helmet, ground truth helmet_on_head; small/distant helmets; total color-mask area is below the class threshold.
hsv_color_mask K2: predicted no_helmet, ground truth helmet_on_head; low-light night scene; the frozen color mask area is below the class threshold.
hsv_color_mask K3: predicted no_helmet, ground truth helmet_on_head; small/distant helmets; total color-mask area is below the class threshold.


## 5. Çıktılar, sınırlılıklar ve doğrulama

Bu sonuç küçük bir eğitim prototipidir: yalnızca iki development ve dört control görüntüsü vardır. Sarı renk tek başına nesnenin baret olduğunu kanıtlamaz; sınıf kuralı basit bir başlangıç sezgisidir. Karanlık ve uzak örneklerdeki kaçırmalar gerçek ölçüm olarak korunmuştur. Kutu IoU değerlendirmesi bu deneyin kapsamında değildir.

Bir sonraki kontrol: K2/K3 için eşiği control üzerinde değiştirmeden, yöntem geliştiricileriyle aydınlatma/ölçek duyarlılığına yönelik ayrı deney tasarlamak ve ikinci bir anotatörle uzak baş kutularını gözden geçirmek.

`individual_notes/nesli.md` ve `docs/ai_usage.md` ölçüm kapsamı ile sınırlılıkları kaydeder. Ortak şema doğrulaması `scripts/validate_project.py --require-data` ile yapılır.


In [5]:
preview_files = sorted((ROOT / "annotations/previews").glob("*_ground_truth.png"))
mask_files = sorted((ROOT / "results/masks/rgb").glob("*_rgb_baseline.png"))
mask_files += sorted((ROOT / "results/masks/hsv").glob("*_hsv_baseline.png"))
print(f"Ground truth previews: {len(preview_files)}")
print(f"Control masks: {len(mask_files)}")
assert len(preview_files) == 6 and len(mask_files) == 8


Ground truth previews: 6
Control masks: 8
